# Fine-tuning de DistilBERT para clasificación multietiqueta de síntomas depresivos

Sistema Inteligente para la Identificación de Problemas de Salud Mental Juvenil en RRSS
**Trabajo Fin de Grado**


### Introducción

Este notebook entrena el modelo de clasificación multietiqueta que constituye el núcleo del
sistema descrito en la memoria (capítulo 4). A partir de un conjunto de publicaciones de Reddit
etiquetadas semi-automáticamente por síntoma (según los criterios diagnósticos del DSM-5 para
el trastorno depresivo mayor), se ajusta (fine-tuning) el modelo preentrenado **DistilBERT**
para predecir, dado un texto, la presencia o ausencia de cada uno de los diez síntomas.

#### Estructura del notebook

1. [Importación de librerías y configuración del entorno](#section01)
2. [Carga y preprocesamiento de los datos](#section02)
3. [Preparación del Dataset y del Dataloader](#section03)
4. [Definición de la red neuronal (DistilBERTClass)](#section04)
5. [Entrenamiento (fine-tuning) del modelo](#section05)
6. [Evaluación del modelo sobre el conjunto de test](#section06)
7. [Guardado del modelo y artefactos para inferencia](#section07)

#### Detalles técnicos

- **Datos**: publicaciones del subreddit `r/depression` (y subreddits neutrales usados como
  ejemplos negativos), etiquetadas para diez síntomas del trastorno depresivo mayor
  (`sintoma_1` a `sintoma_10`), tal como se describe en la sección 4.2 de la memoria.
- **Modelo base**: `distilbert-base-uncased` (Hugging Face `transformers`).
- **Tarea**: clasificación multietiqueta (cada texto puede presentar 0, 1 o varios síntomas
  simultáneamente), por lo que se emplea `BCEWithLogitsLoss` en lugar de entropía cruzada
  estándar, y se evalúa con métricas específicas de multietiqueta (F1 micro/macro, Hamming
  Loss/Score, ROC-AUC por clase) en lugar de accuracy simple.
- **Requisitos**: Python ≥ 3.9, PyTorch, `transformers`, `scikit-learn`, GPU recomendada.


<a id='section01'></a>
## 1. Importación de librerías y configuración del entorno

Se importan las librerías necesarias (manejo de datos, PyTorch, `transformers`, métricas de
scikit-learn) y se selecciona el dispositivo de cómputo (GPU si está disponible).


In [ ]:
!pip install -q "transformers>=4.40" torch scikit-learn matplotlib seaborn tqdm


In [ ]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
from torch import cuda
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import DistilBertTokenizer, DistilBertModel
import logging as hf_logging

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score, roc_auc_score,
    roc_curve, auc, multilabel_confusion_matrix
)

warnings.simplefilter("ignore")
hf_logging.basicConfig(level=hf_logging.ERROR)


In [ ]:
device = "cuda" if cuda.is_available() else "cpu"
print(f"Dispositivo de cómputo: {device}")


<a id='section02'></a>
## 2. Carga y preprocesamiento de los datos

Se parte del conjunto `train.csv` generado durante la fase de construcción del dataset
(sección 4.2 de la memoria): publicaciones de Reddit con una columna de texto (`texto`) y
diez columnas binarias `sintoma_1` ... `sintoma_10` que indican la presencia de cada síntoma.

Ajusta `DATA_PATH` a la ubicación del archivo en tu entorno.


In [ ]:
DATA_PATH = "data/train.csv"  # ruta al csv con las publicaciones y etiquetas por síntoma

SYMPTOM_COLUMNS = [f"sintoma_{i}" for i in range(1, 11)]

ETIQUETAS_SINTOMAS = [
    "Ánimo depresivo persistente",
    "Pérdida de interés o placer (anhedonia)",
    "Alteraciones en el apetito o peso corporal",
    "Trastornos del sueño (insomnio o hipersomnia)",
    "Agitación o enlentecimiento psicomotor",
    "Fatiga o pérdida de energía",
    "Culpabilidad excesiva o autocrítica intensa",
    "Dificultades de concentración o toma de decisiones",
    "Conductas autolesivas",
    "Ideación suicida",
]

data = pd.read_csv(DATA_PATH)
data[SYMPTOM_COLUMNS] = data[SYMPTOM_COLUMNS].fillna(0)

df = pd.DataFrame()
df["text"] = data["texto"]
df["labels"] = data[SYMPTOM_COLUMNS].values.tolist()

df.head()


<a id='section03'></a>
## 3. Preparación del Dataset y del Dataloader

Se definen los hiperparámetros de tokenización/entrenamiento y la clase `MultiLabelDataset`,
que tokeniza cada texto con el tokenizer de DistilBERT y devuelve los tensores (`ids`, `mask`,
`token_type_ids`) junto con el vector de etiquetas (`targets`).

El conjunto se divide en **entrenamiento (80 %)**, **validación (10 %)** y **test (10 %)**,
siguiendo el esquema descrito en la sección 4.2.1.3 de la memoria.


In [ ]:
# Hiperparámetros de la configuración final (sección 4.2.1.3 de la memoria)
MAX_LEN = 128
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 3e-5
DROPOUT = 0.3
WEIGHT_DECAY = 0.01

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased", do_lower_case=True
)


In [ ]:
class MultiLabelDataset(Dataset):
    """Dataset que tokeniza texto y expone (ids, mask, token_type_ids, targets)
    listos para alimentar a DistilBERTClass."""

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.text = dataframe.text
        self.targets = dataframe.labels
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = " ".join(str(self.text[index]).split())

        inputs = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "ids": inputs["input_ids"].squeeze(0),
            "mask": inputs["attention_mask"].squeeze(0),
            "token_type_ids": inputs["token_type_ids"].squeeze(0),
            "targets": torch.tensor(self.targets[index], dtype=torch.float),
        }


In [ ]:
# División 80% train / 10% validación / 10% test
train_data, temp_data = train_test_split(df, test_size=0.2, random_state=200)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=200)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"Conjunto completo:     {df.shape}")
print(f"Entrenamiento (80%):   {train_data.shape}")
print(f"Validación (10%):      {val_data.shape}")
print(f"Test (10%):            {test_data.shape}")

training_set = MultiLabelDataset(train_data, tokenizer, MAX_LEN)
validation_set = MultiLabelDataset(val_data, tokenizer, MAX_LEN)
testing_set = MultiLabelDataset(test_data, tokenizer, MAX_LEN)

train_params = {"batch_size": TRAIN_BATCH_SIZE, "shuffle": True, "num_workers": 0}
eval_params = {"batch_size": VALID_BATCH_SIZE, "shuffle": True, "num_workers": 0}

training_loader = DataLoader(training_set, **train_params)
validation_loader = DataLoader(validation_set, **eval_params)
testing_loader = DataLoader(testing_set, **eval_params)


<a id='section04'></a>
## 4. Definición de la red neuronal (`DistilBERTClass`)

Arquitectura descrita en la sección 4.2.1.2 de la memoria: el encoder de DistilBERT,
seguido de una capa lineal (`768 → 768`) con activación `Tanh`, `Dropout` para
regularización, y una capa de salida lineal (`768 → 10`) que produce los logits para
los diez síntomas.

- **Pérdida**: `BCEWithLogitsLoss` (una sigmoide + entropía cruzada binaria por síntoma),
  adecuada para clasificación multietiqueta.
- **Optimizador**: `AdamW`, separando los parámetros con y sin `weight_decay` (no se
  penalizan los sesgos ni los pesos de `LayerNorm`), tal como se ajustó en la
  configuración final del modelo (weight decay = 0.01).


In [ ]:
class DistilBERTClass(torch.nn.Module):
    def __init__(self, num_labels=10, dropout=DROPOUT, freeze_base_model=False):
        super().__init__()
        self.base_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

        if freeze_base_model:
            for param in self.base_model.parameters():
                param.requires_grad = False

        self.pre_classifier = torch.nn.Linear(768, 768)
        self.activation = torch.nn.Tanh()
        self.dropout = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        pooled_output = hidden_state[:, 0]  # token [CLS]

        x = self.pre_classifier(pooled_output)
        x = self.activation(x)
        x = self.dropout(x)
        return self.classifier(x)


model = DistilBERTClass()
model.to(device)


In [ ]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)


# Separar parámetros con y sin weight decay (no se penalizan bias ni LayerNorm)
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "bias" in name or "LayerNorm.weight" in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)


<a id='section05'></a>
## 5. Entrenamiento (fine-tuning) del modelo

Se define un identificador de la configuración (`model_id`) y un diccionario `history`
que acumula, por época, la pérdida y las métricas de entrenamiento y validación
(exactitud, F1 micro, F1 macro y Hamming Loss), siguiendo el criterio de selección de
época descrito en la sección 4.2.1.3 de la memoria (early-stopping basado en F1
micro/macro y Hamming Loss de validación).


In [ ]:
model_id = (
    f"DistilBERTClass_dropout{DROPOUT}_wd{WEIGHT_DECAY}"
    f"_len{MAX_LEN}_bs{TRAIN_BATCH_SIZE}_ep{EPOCHS}_lr{LEARNING_RATE}"
)

history = {
    "train_loss": [], "val_loss": [],
    "train_accuracy": [], "val_accuracy": [],
    "train_f1_micro": [], "val_f1_micro": [],
    "train_f1_macro": [], "val_f1_macro": [],
    "train_hamming_loss": [], "val_hamming_loss": [],
    "model_id": model_id,
}


In [ ]:
def evaluate(loader):
    """Evalúa el modelo sobre un dataloader y devuelve pérdida y métricas multietiqueta."""
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for data in loader:
            ids = data["ids"].to(device, dtype=torch.long)
            mask = data["mask"].to(device, dtype=torch.long)
            token_type_ids = data["token_type_ids"].to(device, dtype=torch.long)
            targets = data["targets"].to(device, dtype=torch.float)

            outputs = model(ids, mask, token_type_ids)
            loss = loss_fn(outputs, targets)
            running_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.int().cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    return {
        "val_loss": running_loss / len(loader),
        "accuracy": (y_pred == y_true).mean(),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


In [ ]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for data in tqdm(training_loader, desc=f"Época {epoch + 1}/{EPOCHS}"):
        ids = data["ids"].to(device, dtype=torch.long)
        mask = data["mask"].to(device, dtype=torch.long)
        token_type_ids = data["token_type_ids"].to(device, dtype=torch.long)
        targets = data["targets"].to(device, dtype=torch.float)

        outputs = model(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
        all_preds.append(preds)
        all_targets.append(targets.int().cpu().numpy())

    y_pred_train = np.vstack(all_preds)
    y_true_train = np.vstack(all_targets)

    train_metrics = {
        "loss": running_loss / len(training_loader),
        "accuracy": (y_pred_train == y_true_train).mean(),
        "f1_micro": f1_score(y_true_train, y_pred_train, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true_train, y_pred_train, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true_train, y_pred_train),
    }
    val_metrics = evaluate(validation_loader)

    history["train_loss"].append(train_metrics["loss"])
    history["train_accuracy"].append(train_metrics["accuracy"])
    history["train_f1_micro"].append(train_metrics["f1_micro"])
    history["train_f1_macro"].append(train_metrics["f1_macro"])
    history["train_hamming_loss"].append(train_metrics["hamming_loss"])

    history["val_loss"].append(val_metrics["val_loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_f1_micro"].append(val_metrics["f1_micro"])
    history["val_f1_macro"].append(val_metrics["f1_macro"])
    history["val_hamming_loss"].append(val_metrics["hamming_loss"])

    print(
        f"Época {epoch + 1}: "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_f1_micro={train_metrics['f1_micro']:.4f} "
        f"train_f1_macro={train_metrics['f1_macro']:.4f} | "
        f"val_loss={val_metrics['val_loss']:.4f} "
        f"val_f1_micro={val_metrics['f1_micro']:.4f} "
        f"val_f1_macro={val_metrics['f1_macro']:.4f} "
        f"val_hamming_loss={val_metrics['hamming_loss']:.4f}"
    )


In [ ]:
for epoch in range(EPOCHS):
    train_one_epoch(epoch)


### Curvas de entrenamiento y validación

Se grafica la evolución de la pérdida y de las métricas de validación por época. Como se
indica en la memoria, la época seleccionada como configuración final fue la **cuarta**,
punto en el que se alcanza un buen equilibrio entre F1 micro/macro y Hamming Loss en
validación, evitando el sobreajuste observado en épocas posteriores.


In [ ]:
os.makedirs(model_id, exist_ok=True)
epochs_range = range(1, len(history["train_loss"]) + 1)

metric_labels = [
    ("loss", "Pérdida"),
    ("accuracy", "Exactitud"),
    ("f1_micro", "F1 Micro"),
    ("f1_macro", "F1 Macro"),
    ("hamming_loss", "Hamming Loss"),
]

for metric_key, metric_name in metric_labels:
    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, history[f"train_{metric_key}"], label=f"{metric_name} (entrenamiento)")
    plt.plot(epochs_range, history[f"val_{metric_key}"], label=f"{metric_name} (validación)")
    plt.xlabel("Época")
    plt.ylabel(metric_name)
    plt.title(f"{metric_name} por época")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{model_id}/{metric_key}_por_epoca.png")
    plt.show()


In [ ]:
def hamming_score(y_true, y_pred):
    """Proporción media de solapamiento entre etiquetas reales y predichas por instancia."""
    scores = []
    for i in range(y_true.shape[0]):
        set_true = set(np.where(y_true[i])[0])
        set_pred = set(np.where(y_pred[i])[0])
        if not set_true and not set_pred:
            scores.append(1.0)
        else:
            scores.append(len(set_true & set_pred) / len(set_true | set_pred))
    return np.mean(scores)


In [ ]:
# Guardado de la configuración y del historial de entrenamiento

def append_json(filename, new_entry):
    """Añade una entrada a un fichero json que acumula una lista de registros."""
    records = []
    if os.path.exists(filename):
        with open(filename, "r") as f:
            records = json.load(f)
    records.append(new_entry)
    with open(filename, "w") as f:
        json.dump(records, f, indent=4)


config = {
    "MAX_LEN": MAX_LEN,
    "TRAIN_BATCH_SIZE": TRAIN_BATCH_SIZE,
    "VALID_BATCH_SIZE": VALID_BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "DROPOUT": DROPOUT,
    "WEIGHT_DECAY": WEIGHT_DECAY,
}

append_json("model_config.json", config)
append_json("training_history.json", history)


<a id='section06'></a>
## 6. Evaluación del modelo sobre el conjunto de test

Se evalúa el modelo sobre el conjunto de **test** (10 % de los datos, no utilizado en
entrenamiento ni en validación) mediante:

- **Hamming Score / Hamming Loss**: comparación directa entre etiquetas reales y predichas.
- **Exact Match Ratio, precisión, recall, F1 (micro/macro) y ROC-AUC**: conjunto de métricas
  estándar para clasificación multietiqueta (sección 2.3.4.4 de la memoria).
- **Curvas ROC y matrices de confusión por síntoma**, para analizar el comportamiento del
  modelo en cada una de las diez clases.


In [ ]:
def run_inference(loader):
    """Ejecuta el modelo sobre un dataloader y devuelve probabilidades y etiquetas reales."""
    model.eval()
    all_targets, all_outputs = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc="Evaluando"):
            ids = data["ids"].to(device, dtype=torch.long)
            mask = data["mask"].to(device, dtype=torch.long)
            token_type_ids = data["token_type_ids"].to(device, dtype=torch.long)
            targets = data["targets"].to(device, dtype=torch.float)

            outputs = model(ids, mask, token_type_ids)

            all_targets.extend(targets.cpu().numpy().tolist())
            all_outputs.extend(torch.sigmoid(outputs).cpu().numpy().tolist())
    return np.array(all_outputs), np.array(all_targets)


test_probs, test_targets = run_inference(testing_loader)
test_preds = test_probs >= 0.5


In [ ]:
val_hamming_loss = hamming_loss(test_targets, test_preds)
val_hamming_score = hamming_score(test_targets, test_preds)

print(f"Hamming Score = {val_hamming_score:.4f}")
print(f"Hamming Loss  = {val_hamming_loss:.4f}")


In [ ]:
def evaluate_metrics(y_true, y_pred, y_prob=None, average="macro"):
    """Calcula el conjunto de métricas estándar para clasificación multietiqueta."""
    metrics = {
        "exact_match_ratio": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
    }
    if y_prob is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob, average=average)
        except ValueError:
            metrics["roc_auc"] = None  # puede fallar si alguna clase está ausente
    return metrics


test_metrics = evaluate_metrics(test_targets, test_preds, test_probs, average="macro")
for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}" if value is not None else f"{name}: N/A")


In [ ]:
def plot_roc_curves(y_true, y_prob, class_names, save_dir):
    n_classes = y_true.shape[1]
    plt.figure(figsize=(10, 7))
    auc_by_class = {}

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        auc_by_class[i] = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"{class_names[i]} (AUC = {auc_by_class[i]:.2f})")

    plt.plot([0, 1], [0, 1], "k--", lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("Tasa de falsos positivos")
    plt.ylabel("Tasa de verdaderos positivos")
    plt.title("Curvas ROC por síntoma")
    plt.legend(loc="lower right", fontsize="small")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "roc_por_sintoma.png"))
    plt.show()

    return auc_by_class


auc_by_class = plot_roc_curves(test_targets, test_probs, ETIQUETAS_SINTOMAS, model_id)


In [ ]:
# AUC por síntoma, ordenado de mayor a menor
auc_df = pd.DataFrame(
    {"sintoma": ETIQUETAS_SINTOMAS, "AUC": [auc_by_class[i] for i in range(len(ETIQUETAS_SINTOMAS))]}
).sort_values(by="AUC", ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(auc_df["sintoma"], auc_df["AUC"], color="skyblue")
plt.xlabel("AUC")
plt.title("AUC por síntoma")
plt.xlim(0.5, 1.0)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(f"{model_id}/auc_por_sintoma.png")
plt.show()


In [ ]:
def plot_confusion_matrices(y_true, y_pred, class_names, save_dir):
    """Dibuja y guarda la matriz de confusión de cada síntoma por separado."""
    conf_matrices = multilabel_confusion_matrix(y_true, y_pred)

    for i, cm in enumerate(conf_matrices):
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
        plt.title(f"Matriz de confusión — {class_names[i]}")
        plt.xlabel("Predicción")
        plt.ylabel("Real")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"matriz_confusion_sintoma_{i + 1}.png"))
        plt.show()


plot_confusion_matrices(test_targets, test_preds, ETIQUETAS_SINTOMAS, model_id)


<a id='section07'></a>
## 7. Guardado del modelo y artefactos para inferencia

Se guardan el modelo entrenado, el vocabulario del tokenizer y las métricas finales de
test, para su uso posterior en el backend de la aplicación web (`ml/predict.py`, sección
4.3.2 de la memoria).


In [ ]:
output_model_file = f"./{model_id}/pytorch_distilbert_depression.bin"
output_vocab_file = f"./{model_id}/vocab_distilbert_depression"

os.makedirs(model_id, exist_ok=True)
torch.save(model.state_dict(), output_model_file)
tokenizer.save_vocabulary(output_vocab_file)

print(f"Modelo guardado en: {output_model_file}")
print(f"Vocabulario guardado en: {output_vocab_file}")


In [ ]:
final_metrics = evaluate_metrics(test_targets, test_preds, test_probs, average="macro")
final_metrics["hamming_loss"] = val_hamming_loss
final_metrics["hamming_score"] = val_hamming_score
final_metrics["model_id"] = model_id

append_json("test_metrics_history.json", final_metrics)
final_metrics
